# 02_SOLVER

Solver KRONA / LTP - V0 controlada.

Escopo desta versão:

- Primeiro mês apenas.
- Otimização somente por `ALOC_REC`.
- `COD_FER_UNID` será tratado em etapa futura.
- Entrada única: `bd_LTP_NEC_SOLVER.xlsx`.
- Demanda do item: `NEC_PCS`.
- Alternativas produtivas: combinação de `ID_PROD_UNID_FAT`, `PRIOR_MATPAR`, `PRIOR_ROT` e `ALOC_REC`.
- Menor `PRIOR_MATPAR` e menor `PRIOR_ROT` são melhores, mas não bloqueiam o uso de alternativas.
- Capacidade oficial: `HOR_CAP`.
- `HOR_REC`, `HOR_FER` e `HOR_CAP` são preservadas para auditoria.
- Não atendimento é reportado somente em peças.


In [ ]:
# ============================================================
# 00. CHECKPOINT DA VERSAO
# ============================================================

CHECKPOINT_02_SOLVER = "02_SOLVER_CONSOLIDADO_HOR_CAP_NAO_ATEND_ITEM_20260621"
print(CHECKPOINT_02_SOLVER)


In [ ]:
# ============================================================
# 01. IMPORTS E CAMINHOS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from openpyxl import load_workbook

# Caminhos oficiais no ambiente do usuário
OUTPUT_DIR = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")
INPUT_FILE = OUTPUT_DIR / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE = OUTPUT_DIR / "bd_SOLVER_ALOC_REC_V0.xlsx"

# Fallback para teste fora da máquina do usuário, por exemplo no ambiente ChatGPT
if not INPUT_FILE.exists():
    fallback_input = Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx")
    if fallback_input.exists():
        INPUT_FILE = fallback_input
        OUTPUT_SOLVER_FILE = Path("/mnt/data/bd_SOLVER_ALOC_REC_V0.xlsx")

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_SOLVER_FILE:", OUTPUT_SOLVER_FILE)

# Flags usados quando a base ainda não trouxer NEC_PCS pronto.
LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True


In [ ]:
# ============================================================
# 02. LEITURA DA BASE
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Arquivo de entrada não encontrado: {INPUT_FILE}")

bd = pd.read_excel(INPUT_FILE)

print("Base carregada:", bd.shape)
display(bd.head())


In [ ]:
# ============================================================
# 03. FUNCOES AUXILIARES
# ============================================================

def validar_colunas_obrigatorias(df: pd.DataFrame) -> None:
    obrigatorias = [
        "MES_REF",
        "ID_PROD_UNID_FAT",
        "ALOC_REC",
        "PRIOR_MATPAR",
        "PRIOR_ROT",
        "PCS_HORA",
        "HOR_REC",
        "HOR_FER",
    ]
    faltantes = [c for c in obrigatorias if c not in df.columns]
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes na base: {faltantes}")


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def calcular_hor_cap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["HOR_REC"] = to_num(df["HOR_REC"]).fillna(0)
    df["HOR_FER"] = to_num(df["HOR_FER"]).fillna(0)
    df["HOR_CAP"] = np.where(
        df["HOR_FER"] <= 0,
        df["HOR_REC"],
        df[["HOR_REC", "HOR_FER"]].min(axis=1)
    )
    df["REGRA_HOR_CAP"] = np.where(
        df["HOR_FER"] <= 0,
        "HOR_FER <= 0: HOR_CAP = HOR_REC",
        "HOR_CAP = menor entre HOR_REC e HOR_FER"
    )
    return df


def colunas_existentes(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]


In [ ]:
# ============================================================
# 04. PREPARACAO DA BASE DO PRIMEIRO MES
# ============================================================

validar_colunas_obrigatorias(bd)

bd = bd.copy()
bd["MES_REF"] = pd.to_datetime(bd["MES_REF"], errors="coerce")

for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER"]:
    bd[c] = to_num(bd[c])

# Se a base ainda não trouxe NEC_PCS, calcula com a regra do processo anterior.
if "NEC_PCS" not in bd.columns:
    cols_calc = [
        "LTP_CART_ARR_MES_ANT",
        "LTP_CART_MES_ATUAL",
        "LTP_SALDO_PREV_PCS",
        "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS",
        "LTP_EST_TRANS_PCS",
        "ORI_TOT_PCS",
        "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS",
        "LTP_COMP_NEC_PCS",
        "LIMIT_PCS",
        "LOTE_MIN",
        "QTD_EMB",
    ]
    for c in cols_calc:
        if c not in bd.columns:
            bd[c] = 0
        bd[c] = to_num(bd[c]).fillna(0)

    base_comum = (
        bd["LTP_CART_ARR_MES_ANT"]
        + bd["LTP_CART_MES_ATUAL"]
        + bd["LTP_SALDO_PREV_PCS"]
        + bd["LTP_EST_SEG_PCS"]
        - bd["LTP_EST_INI_PCS"]
        - bd["LTP_EST_TRANS_PCS"]
        - bd["ORI_TOT_PCS"]
        - bd["TRIANG_TOT_PCS"]
    )

    bd["NEC_PCS"] = np.where(
        bd["MESMA_REG"].astype(str).str.upper().eq("NAO"),
        base_comum + bd["LTP_SALDO_PREV_PROX_MES_PCS"],
        base_comum,
    )

    bd["NEC_PCS"] = bd["NEC_PCS"].clip(lower=0)
    bd["NEC_PCS"] = bd["NEC_PCS"] + bd["LTP_COMP_NEC_PCS"]
    bd["NEC_PCS"] = np.maximum(bd["NEC_PCS"], bd["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = bd["NEC_PCS"].gt(0) & bd["LOTE_MIN"].gt(0)
        bd.loc[mask_lote, "NEC_PCS"] = np.maximum(
            bd.loc[mask_lote, "NEC_PCS"],
            bd.loc[mask_lote, "LOTE_MIN"]
        )

    if MULTIPLO_EMB_FLAG:
        mask_emb = (
            bd["NEC_PCS"].gt(0)
            & bd["QTD_EMB"].gt(0)
            & bd.get("TIPO_PROD", "").astype(str).str.upper().isin(["PA", "MR"])
        )
        bd.loc[mask_emb, "NEC_PCS"] = (
            np.ceil(bd.loc[mask_emb, "NEC_PCS"] / bd.loc[mask_emb, "QTD_EMB"])
            * bd.loc[mask_emb, "QTD_EMB"]
        )

    print("NEC_PCS calculado no notebook porque não existia na base de entrada.")
else:
    bd["NEC_PCS"] = to_num(bd["NEC_PCS"]).fillna(0)
    print("NEC_PCS lido da base de entrada.")

bd = calcular_hor_cap(bd)

primeiro_mes = bd["MES_REF"].min()
bd_mes = bd.loc[bd["MES_REF"].eq(primeiro_mes)].copy()

print("Primeiro MES_REF:", primeiro_mes)
print("bd_mes:", bd_mes.shape)


In [ ]:
# ============================================================
# 05. DEMANDA UNICA POR ITEM
# NEC_PCS pertence ao item, não à rota.
# ============================================================

meta_item_cols = colunas_existentes(
    bd_mes,
    [
        "MES_REF",
        "ID_PROD_UNID_FAT",
        "COD_PROD",
        "DESC_PROD",
        "UNID_PROD",
        "UNID_FAT",
        "TIPO_PROD",
        "NEC_PCS",
    ]
)

bd_demanda_solver = (
    bd_mes
    .sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    [meta_item_cols]
    .drop_duplicates(subset=["MES_REF", "ID_PROD_UNID_FAT"], keep="first")
    .copy()
)

bd_demanda_solver["NEC_PCS"] = to_num(bd_demanda_solver["NEC_PCS"]).fillna(0)
bd_demanda_solver = bd_demanda_solver.loc[bd_demanda_solver["NEC_PCS"] > 0].copy()

print("bd_demanda_solver:", bd_demanda_solver.shape)
display(bd_demanda_solver.head())


In [ ]:
# ============================================================
# 06. ALTERNATIVAS PRODUTIVAS POR ITEM / ALOC_REC
# PCS_HORA pertence à alternativa produtiva/recurso.
# ============================================================

alt_cols_base = [
    "MES_REF",
    "ID_PROD_UNID_FAT",
    "COD_PROD",
    "DESC_PROD",
    "UNID_PROD",
    "UNID_FAT",
    "TIPO_PROD",
    "PRIOR_MATPAR",
    "PRIOR_ROT",
    "CLASS_ROT",
    "ALOC_REC",
    "COD_FER_UNID",
    "ID_RECURSO",
    "ID_FERRAMENTA",
    "PCS_HORA",
    "HOR_REC",
    "HOR_FER",
    "HOR_CAP",
    "REGRA_HOR_CAP",
]

alt_cols = colunas_existentes(bd_mes, alt_cols_base)

bd_alternativas_solver = bd_mes.loc[
    bd_mes["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])
    & bd_mes["ALOC_REC"].notna()
    & bd_mes["PCS_HORA"].gt(0)
    & bd_mes["HOR_CAP"].gt(0),
    alt_cols
].copy()

# remove duplicidades exatas de alternativa
chaves_alt = colunas_existentes(
    bd_alternativas_solver,
    [
        "MES_REF",
        "ID_PROD_UNID_FAT",
        "PRIOR_MATPAR",
        "PRIOR_ROT",
        "ALOC_REC",
        "COD_FER_UNID",
        "PCS_HORA",
    ]
)
bd_alternativas_solver = bd_alternativas_solver.drop_duplicates(subset=chaves_alt).copy()

bd_alternativas_solver["PRIOR_MATPAR"] = to_num(bd_alternativas_solver["PRIOR_MATPAR"]).fillna(999999)
bd_alternativas_solver["PRIOR_ROT"] = to_num(bd_alternativas_solver["PRIOR_ROT"]).fillna(999999)
bd_alternativas_solver["SCORE_PRIORIDADE"] = (
    bd_alternativas_solver["PRIOR_MATPAR"] * 100
    + bd_alternativas_solver["PRIOR_ROT"]
)

print("bd_alternativas_solver:", bd_alternativas_solver.shape)
display(bd_alternativas_solver.head())


In [ ]:
# ============================================================
# 07. SOLVER ALOC_REC V0
# Variáveis:
#   x[item, alternativa] = QTD_PRODUZIR_SOLVER
#   nao_atend[item] = QTD_NAO_ATEND_SOLVER
# Restrição de demanda:
#   soma(x) + nao_atend = NEC_PCS
# Restrição de capacidade:
#   soma(x / PCS_HORA) <= HOR_CAP por ALOC_REC
# ============================================================

PESO_NAO_ATEND = 100_000.0
PESO_USO_ALTERNATIVO = 1.0
TOL_SOLVER = 1e-7


def resolver_solver_aloc_rec_v0(demanda: pd.DataFrame, alternativas: pd.DataFrame):
    demanda = demanda.reset_index(drop=True).copy()
    alternativas = alternativas.reset_index(drop=True).copy()

    item_to_i = {k: i for i, k in enumerate(demanda["ID_PROD_UNID_FAT"])}
    alternativas = alternativas.loc[alternativas["ID_PROD_UNID_FAT"].isin(item_to_i)].copy()
    alternativas["ALOC_REC_STR"] = alternativas["ALOC_REC"].astype(str)

    recs = alternativas["ALOC_REC_STR"].sort_values().unique().tolist()
    rec_to_i = {k: i for i, k in enumerate(recs)}

    n_alt = len(alternativas)
    n_item = len(demanda)
    n_var = n_alt + n_item

    c = np.zeros(n_var, dtype=float)
    c[:n_alt] = alternativas["SCORE_PRIORIDADE"].to_numpy(dtype=float) * PESO_USO_ALTERNATIVO
    c[n_alt:] = PESO_NAO_ATEND

    # demanda: soma produção alternativa + nao_atend = NEC_PCS
    A_eq = np.zeros((n_item, n_var), dtype=float)
    b_eq = demanda["NEC_PCS"].to_numpy(dtype=float)

    for j, row in alternativas.iterrows():
        A_eq[item_to_i[row["ID_PROD_UNID_FAT"]], j] = 1.0

    for i in range(n_item):
        A_eq[i, n_alt + i] = 1.0

    # capacidade por ALOC_REC
    A_ub = np.zeros((len(recs), n_var), dtype=float)
    for j, row in alternativas.iterrows():
        A_ub[rec_to_i[row["ALOC_REC_STR"]], j] = 1.0 / float(row["PCS_HORA"])

    cap_solver_df = (
        alternativas
        .groupby("ALOC_REC_STR", as_index=False)
        .agg(
            HOR_CAP=("HOR_CAP", "min"),
            HOR_REC=("HOR_REC", "max"),
            HOR_FER=("HOR_FER", "min"),
        )
    )

    b_ub = cap_solver_df.set_index("ALOC_REC_STR").loc[recs, "HOR_CAP"].to_numpy(dtype=float)

    bounds = [(0, None)] * n_var

    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    return res, alternativas, demanda, recs, cap_solver_df


res, alt_model, demanda_model, recs_solver, cap_solver_df = resolver_solver_aloc_rec_v0(
    bd_demanda_solver,
    bd_alternativas_solver,
)

print("Status:", res.status)
print("Mensagem:", res.message)
print("Objetivo:", res.fun if res.success else None)

if not res.success:
    raise RuntimeError(f"Solver não encontrou solução ótima. Status={res.status}. Mensagem={res.message}")


In [ ]:
# ============================================================
# 08. MONTAGEM DAS SAIDAS DE NEGOCIO
# ============================================================

n_alt = len(alt_model)
x = res.x[:n_alt]
nao_atend = res.x[n_alt:]

# 02_PLANO_PRODUCAO
plano_cols = colunas_existentes(
    alt_model,
    [
        "MES_REF",
        "ID_PROD_UNID_FAT",
        "COD_PROD",
        "DESC_PROD",
        "UNID_PROD",
        "UNID_FAT",
        "TIPO_PROD",
        "ALOC_REC",
        "COD_FER_UNID",
        "PRIOR_MATPAR",
        "PRIOR_ROT",
        "CLASS_ROT",
        "PCS_HORA",
        "HOR_REC",
        "HOR_FER",
        "HOR_CAP",
        "REGRA_HOR_CAP",
        "SCORE_PRIORIDADE",
    ]
)

plano_producao = alt_model[plano_cols].copy()
plano_producao["QTD_PRODUZIR_SOLVER"] = x
plano_producao["HR_PRODUZIR_SOLVER"] = plano_producao["QTD_PRODUZIR_SOLVER"] / plano_producao["PCS_HORA"]
plano_producao = plano_producao.loc[plano_producao["QTD_PRODUZIR_SOLVER"] > TOL_SOLVER].copy()
plano_producao = plano_producao.sort_values(["ALOC_REC", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])

# 03_NAO_ATEND_ITEM
atendido_item = (
    plano_producao
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"))
)

nao_atend_item = demanda_model.copy()
nao_atend_item = nao_atend_item.merge(
    atendido_item,
    on=["MES_REF", "ID_PROD_UNID_FAT"],
    how="left",
)
nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]
nao_atend_item.loc[nao_atend_item["QTD_NAO_ATEND_SOLVER"].abs() < TOL_SOLVER, "QTD_NAO_ATEND_SOLVER"] = 0

# deixa somente colunas de negócio. Não atendimento somente em peças.
nao_atend_cols = colunas_existentes(
    nao_atend_item,
    [
        "MES_REF",
        "ID_PROD_UNID_FAT",
        "COD_PROD",
        "DESC_PROD",
        "UNID_PROD",
        "UNID_FAT",
        "TIPO_PROD",
        "NEC_PCS",
        "QTD_ATENDIDA_SOLVER",
        "QTD_NAO_ATEND_SOLVER",
    ]
)
nao_atend_item = nao_atend_item[nao_atend_cols].sort_values("QTD_NAO_ATEND_SOLVER", ascending=False)

# 01_RESUMO_MAQUINAS
prod_rec = (
    plano_producao
    .groupby("ALOC_REC", as_index=False)
    .agg(
        HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"),
        QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"),
        QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"),
        QTD_ROTAS=("ID_PROD_UNID_FAT", "count"),
    )
)

cap_resumo = cap_solver_df.rename(columns={"ALOC_REC_STR": "ALOC_REC"}).copy()
resumo_maquinas = cap_resumo.merge(prod_rec, on="ALOC_REC", how="left")
for c in ["HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_ROTAS"]:
    resumo_maquinas[c] = resumo_maquinas[c].fillna(0)

resumo_maquinas["OCUPACAO_SOLVER_PCT"] = np.where(
    resumo_maquinas["HOR_CAP"] > 0,
    resumo_maquinas["HR_PRODUZIR_SOLVER"] / resumo_maquinas["HOR_CAP"],
    0,
)
resumo_maquinas["HR_OCIOSA_SOLVER"] = resumo_maquinas["HOR_CAP"] - resumo_maquinas["HR_PRODUZIR_SOLVER"]
resumo_maquinas["ESTOURO_HR_SOLVER"] = np.where(
    resumo_maquinas["HR_PRODUZIR_SOLVER"] > resumo_maquinas["HOR_CAP"] + TOL_SOLVER,
    resumo_maquinas["HR_PRODUZIR_SOLVER"] - resumo_maquinas["HOR_CAP"],
    0,
)
resumo_maquinas["STATUS_CAPACIDADE"] = np.where(
    resumo_maquinas["ESTOURO_HR_SOLVER"] > TOL_SOLVER,
    "ESTOURO",
    "OK",
)
resumo_maquinas = resumo_maquinas.sort_values(["OCUPACAO_SOLVER_PCT", "HR_PRODUZIR_SOLVER"], ascending=False)

# 04_AUDITORIA_CAPACIDADE
auditoria_capacidade_cols = colunas_existentes(
    bd_alternativas_solver,
    ["ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP", "REGRA_HOR_CAP"]
)
auditoria_capacidade = (
    bd_alternativas_solver[auditoria_capacidade_cols]
    .drop_duplicates()
    .sort_values(["ALOC_REC", "COD_FER_UNID"])
)

# 05_ALTERNATIVAS_ITEM
alternativas_item = bd_alternativas_solver.copy().sort_values(
    ["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]
)

print("01_RESUMO_MAQUINAS:", resumo_maquinas.shape)
print("02_PLANO_PRODUCAO:", plano_producao.shape)
print("03_NAO_ATEND_ITEM:", nao_atend_item.shape)
print("04_AUDITORIA_CAPACIDADE:", auditoria_capacidade.shape)
print("05_ALTERNATIVAS_ITEM:", alternativas_item.shape)


In [ ]:
# ============================================================
# 09. VALIDACOES
# ============================================================

# fechamento da demanda
nec_total = float(nao_atend_item["NEC_PCS"].sum())
atendido_total = float(nao_atend_item["QTD_ATENDIDA_SOLVER"].sum())
nao_atend_total = float(nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum())
dif_demanda = nec_total - atendido_total - nao_atend_total

# capacidade
qtd_rec_estouro = int((resumo_maquinas["STATUS_CAPACIDADE"] == "ESTOURO").sum())
estouro_hr_total = float(resumo_maquinas["ESTOURO_HR_SOLVER"].sum())

# regra HOR_CAP
val_hor_cap = bd_alternativas_solver.copy()
val_hor_cap["HOR_CAP_ESPERADO"] = np.where(
    val_hor_cap["HOR_FER"] <= 0,
    val_hor_cap["HOR_REC"],
    val_hor_cap[["HOR_REC", "HOR_FER"]].min(axis=1),
)
val_hor_cap["DIF_HOR_CAP"] = val_hor_cap["HOR_CAP"] - val_hor_cap["HOR_CAP_ESPERADO"]
qtd_erros_hor_cap = int((val_hor_cap["DIF_HOR_CAP"].abs() > 1e-4).sum())

validacao = pd.DataFrame([
    {"SECAO": "DEMANDA", "METRICA": "NEC_PCS_TOTAL", "VALOR": nec_total},
    {"SECAO": "DEMANDA", "METRICA": "QTD_ATENDIDA_SOLVER_TOTAL", "VALOR": atendido_total},
    {"SECAO": "DEMANDA", "METRICA": "QTD_NAO_ATEND_SOLVER_TOTAL", "VALOR": nao_atend_total},
    {"SECAO": "DEMANDA", "METRICA": "DIF_FECHAMENTO_DEMANDA", "VALOR": dif_demanda},
    {"SECAO": "CAPACIDADE", "METRICA": "QTD_ALOC_REC", "VALOR": len(resumo_maquinas)},
    {"SECAO": "CAPACIDADE", "METRICA": "QTD_ALOC_REC_ESTOURO", "VALOR": qtd_rec_estouro},
    {"SECAO": "CAPACIDADE", "METRICA": "ESTOURO_HR_TOTAL", "VALOR": estouro_hr_total},
    {"SECAO": "HOR_CAP", "METRICA": "QTD_ERROS_HOR_CAP", "VALOR": qtd_erros_hor_cap},
])

print("VALIDACAO")
display(validacao)

if abs(dif_demanda) > 1e-4:
    raise RuntimeError("Validação de demanda falhou: fechamento diferente de zero.")
if qtd_rec_estouro > 0:
    raise RuntimeError("Validação de capacidade falhou: há ALOC_REC com estouro.")
if qtd_erros_hor_cap > 0:
    raise RuntimeError("Validação HOR_CAP falhou: há linhas com regra divergente.")


In [ ]:
# ============================================================
# 10. EXPORTACAO FINAL
# ============================================================

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)

sheets_to_export = {
    "01_RESUMO_MAQUINAS": resumo_maquinas,
    "02_PLANO_PRODUCAO": plano_producao,
    "03_NAO_ATEND_ITEM": nao_atend_item,
    "04_AUDITORIA_CAPACIDADE": auditoria_capacidade,
    "05_ALTERNATIVAS_ITEM": alternativas_item,
    "99_VALIDACAO": validacao,
}

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    for sheet_name, df_sheet in sheets_to_export.items():
        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        ws.freeze_panes = "A2"
        for col_cells in ws.columns:
            max_len = 0
            col_letter = col_cells[0].column_letter
            for cell in col_cells[:200]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)

print("Arquivo exportado:", OUTPUT_SOLVER_FILE)

wb = load_workbook(OUTPUT_SOLVER_FILE, read_only=True)
print("\nGuias exportadas:")
for aba in wb.sheetnames:
    print("-", aba)
